In [1]:
# Minimal imports – the heavy TensorFlow/Keras stack is unnecessary for this solution
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor




In [2]:
def simple_tfidf_ridge(df_train, df_test, df_submission):
    """
    Fit a TF‑IDF vectorizer on the concatenated text fields and train a Ridge
    regression model for each of the 30 target columns. Predictions are clipped
    to the required [0, 1] range and saved as `submission.csv`.
    """
    # concatenate the three main text columns
    text_train = (
        df_train["question_title"].astype(str)
        + " "
        + df_train["question_body"].astype(str)
        + " "
        + df_train["answer"].astype(str)
    )
    text_test = (
        df_test["question_title"].astype(str)
        + " "
        + df_test["question_body"].astype(str)
        + " "
        + df_test["answer"].astype(str)
    )

    # TF‑IDF on unigrams and bigrams, limited vocab for speed/memory
    vectorizer = TfidfVectorizer(
        max_features=50000, ngram_range=(1, 2), stop_words="english"
    )
    X_train = vectorizer.fit_transform(text_train)
    X_test = vectorizer.transform(text_test)

    # target columns are the 30 label columns after the first column (qa_id)
    target_cols = df_submission.columns[1:]
    y_train = df_train[target_cols].values

    # Use a solver that works with sparse input (lsqr) and avoid CG‑tol issue
    base_model = Ridge(alpha=1.0, solver="lsqr", random_state=42)
    model = MultiOutputRegressor(base_model, n_jobs=-1)
    model.fit(X_train, y_train)

    # Predict and clip to the required range
    y_pred = model.predict(X_test)
    y_pred = np.clip(y_pred, 0.0, 1.0)

    # Build submission DataFrame with correct column order
    outp = {"qa_id": df_test["qa_id"].values}
    for i, col in enumerate(target_cols):
        outp[col] = y_pred[:, i]
    submission = pd.DataFrame(outp)

    # Save to the working directory so Kaggle can find it
    submission.to_csv("submission.csv", index=False)


# Load data from the input directory
df_train = pd.read_csv("/kaggle/input/google-quest-challenge/train.csv")
df_test = pd.read_csv("/kaggle/input/google-quest-challenge/test.csv")
df_submission = pd.read_csv(
    "/kaggle/input/google-quest-challenge/sample_submission.csv"
)

simple_tfidf_ridge(df_train, df_test, df_submission)